In [ ]:
import pandas as pd
import networkx as nx
from pyvis.network import Network
from pathlib import Path
from IPython.display import IFrame

# =============================================================================
# CONFIGURATION
# =============================================================================
TARGET_TEAM_ID = 185651
LOCAL_MATCHES_FILE = Path('../Data/processed/events/master_match_results.csv')
OUTPUT_HTML = "team_network_degree2.html"

# Directory for processed team data as per requirements
PROCESSED_TEAMS_DIR = Path('../Data/processed/teams')

print("--- MISSION START: DEGREE 2 NETWORK VISUALIZATION ---")

# 1. Load and Clean Data
df_matches = pd.read_csv(LOCAL_MATCHES_FILE)
df_matches = df_matches.dropna(subset=['Team_A_ID', 'Team_B_ID'])
df_matches['Team_A_ID'] = df_matches['Team_A_ID'].astype(int)
df_matches['Team_B_ID'] = df_matches['Team_B_ID'].astype(int)

# 2. Identify Tiers (Concentric Circles)
# Degree 0: The Target
degree0 = {TARGET_TEAM_ID}

# Degree 1: Direct Opponents
d1_matches = df_matches[(df_matches['Team_A_ID'].isin(degree0)) | (df_matches['Team_B_ID'].isin(degree0))]
degree1 = set(d1_matches['Team_A_ID']).union(set(d1_matches['Team_B_ID'])) - degree0

# Degree 2: Opponents of Opponents
d2_matches = df_matches[(df_matches['Team_A_ID'].isin(degree1)) | (df_matches['Team_B_ID'].isin(degree1))]
degree2 = set(d2_matches['Team_A_ID']).union(set(d2_matches['Team_B_ID'])) - degree1 - degree0

# Combine all relevant IDs for the graph
all_relevant_ids = degree0.union(degree1).union(degree2)

# Filter matches where BOTH teams are within our 2-degree universe
filtered_matches = df_matches[
    df_matches['Team_A_ID'].isin(all_relevant_ids) & 
    df_matches['Team_B_ID'].isin(all_relevant_ids)
]

# 3. Map Names and Build Graph
id_to_name = pd.concat([
    df_matches[['Team_A_ID', 'Team_A_Name']].rename(columns={'Team_A_ID': 'ID', 'Team_A_Name': 'Name'}),
    df_matches[['Team_B_ID', 'Team_B_Name']].rename(columns={'Team_B_ID': 'ID', 'Team_B_Name': 'Name'})
]).drop_duplicates(subset=['ID']).set_index('ID')['Name'].to_dict()

G = nx.Graph()

# Add Nodes with Tier-based styling
for team_id in all_relevant_ids:
    name = id_to_name.get(team_id, f"ID: {team_id}")
    if team_id in degree0:
        color, size, label = "#ff4d4d", 35, f"TARGET: {name}" # Red
    elif team_id in degree1:
        color, size, label = "#97c2fc", 20, name              # Blue
    else:
        color, size, label = "#dddddd", 10, name              # Grey (Degree 2)
        
    G.add_node(team_id, label=label, title=label, color=color, size=size)

# 4. Add Edges with Weight (Frequency of Play)
edges = filtered_matches.apply(
    lambda x: tuple(sorted([x['Team_A_ID'], x['Team_B_ID']])), axis=1
).value_counts().reset_index()
edges.columns = ['Matchup', 'Weight']

for index, row in edges.iterrows():
    t1, t2 = row['Matchup']
    G.add_edge(t1, t2, value=row['Weight'], title=f"Played {row['Weight']} times")

# 5. Render with Adjusted Physics for Larger Data
print(f"Network Statistics: {len(all_relevant_ids)} Teams | {len(edges)} Relationships")

net = Network(height='750px', width='100%', bgcolor='#ffffff', font_color='black', notebook=True)
net.from_nx(G)

# Barneshut physics is better for Degree 2+ to prevent the 'hairball' effect
net.barnes_hut(gravity=-80000, central_gravity=0.3, spring_length=250, spring_strength=0.001, damping=0.09)

net.show(OUTPUT_HTML)
print(f"[SUCCESS] Interactive network saved to {OUTPUT_HTML}")

IFrame(OUTPUT_HTML, width='100%', height='770px')
